# 02e — Frozen Foundation-Model Head (anggota ensemble ke-5)

**Ide inti**: backbone besar (DINOv2-Large secara default, atau CLIP ViT-B/32) DIBEKUKAN total —
tidak ada backward pass lewat backbone sama sekali. Fitur untuk seluruh 26k gambar train diekstrak
SEKALI (satu forward pass, no_grad), lalu di-cache ke Drive sebagai `.npy`. Yang benar-benar dilatih
per fold hanyalah kepala klasifikasi kecil (MLP 2 layer) di atas fitur yang sudah tetap itu —
karena itu seluruh loop 5 fold selesai dalam hitungan menit, bukan jam seperti MaxViT/SwinV2.

**Kenapa ini relevan untuk masalah yang sudah didiagnosis**: OOF confusion matrix 14-model
menunjukkan 64 dari ~79 kesalahan ensemble ada di batas Organic<->Recyclable (0<->2), dan
`00_Diagnostik_Kemiripan_TrainTest.ipynb` menemukan pola konkret (bunga kertas/tisu, tirai kain,
pipet) yang mirip SECARA VISUAL dengan satu kelas tapi SECARA MATERIAL kelas lain — sesuatu yang
sulit dipisahkan oleh CNN/ViT yang di-fine-tune penuh dengan tujuan klasifikasi 3-kelas kita
sendiri (representasinya terbentuk mengikuti shortcut visual yang cukup untuk 26k data training).
Foundation model besar dilatih dengan objektif yang berbeda skalanya (DINOv2: self-supervised
skala sangat besar, sensitif struktur semantik/part-level; CLIP: kontrastif image-text, caption
manusia sering menyebut material eksplisit) — errornya diharapkan TIDAK berkorelasi dengan error
4 arsitektur lama, sehingga anggota ke-5 ini punya nilai ensemble walau akurasi standalone-nya
mungkin tidak setinggi ConvNeXtV2/SwinV2.

**Batas kepatuhan PRD yang WAJIB dijaga (§2 rule 3, §11)**: notebook ini HANYA membuka gambar dari
`Preprocessing_Output/processed/` (train). Tidak ada satu baris kode pun di sini yang menyentuh
folder `test/` — itu murni hak `04_Inference_and_Submission.ipynb`. Ekstraksi fitur test + inferensi
kepala untuk anggota ke-5 ini SENGAJA ditunda ke pembaruan NB04 di masa depan (belum dibuat di sini)
supaya batas "test hanya boleh disentuh NB04" tidak pernah dilanggar oleh notebook mana pun.

**Catatan label (diperbarui, setara Stage 3.5 tapi diadaptasi ke backbone beku)**: karena head ini
dilatih ulang dari nol tiap fold dalam hitungan menit (bukan lanjut dari checkpoint), tidak perlu
notebook "Stage 3.5" terpisah seperti CNN -- koreksi label & oversampling pattern sulit langsung
diterapkan di SINI, di training aslinya:
- **Training & validasi (early stopping)** memakai label TERKOREKSI (`all_label_corrections.csv`)
  -- 20 gambar yang salah label sekarang dilatih dan dievaluasi dengan label yang benar.
- **Oversampling** (`finetune_stage35_plan.csv`) diterapkan sebagai bobot loss per-sample (bukan
  `WeightedRandomSampler`, karena training di sini full-batch bukan mini-batch) -- gambar pattern
  sulit (mainan_bunga/pipet/tisu/tirai/kain) 3x lebih berpengaruh ke gradien tiap step.
- **`true_label` yang ditulis ke OOF tetap MENTAH** (belum dikoreksi) -- supaya konsisten dengan OOF
  4 arsitektur lain yang saat ini masih berbasis checkpoint Stage 1-3 (label lama), sehingga assersi
  kesamaan `true_label` lintas arsitektur di NB03 tidak pecah. Begitu seluruh tim mengadopsi
  checkpoint Stage 3.5 untuk keempat arsitektur itu, kolom ini juga akan diseragamkan (lihat catatan
  yang sama di `02x_Finetune_Stage35.ipynb`).


In [1]:
# transformers hanya perlu di-install kalau FEATURE_SOURCE='clip_vitb32'; timm (untuk DINOv2) sudah
# lazim terpasang di image Kaggle/Colab tapi kita pin versi yang sama dengan 02x/02a-d untuk
# konsistensi tag model (deviasi dari PRD §12 dicatat di notebook lain, alasan sama: image
# GPU sudah bawa torch/torchvision yang cocok dengan driver CUDA-nya).
!pip install -q timm==1.0.11 transformers==4.46.3


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 74.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import hashlib
import json
import os
import shutil
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 3
N_FOLDS = 5
print("device:", DEVICE, "| torch:", torch.__version__, "| timm:", timm.__version__)


device: cuda | torch: 2.11.0+cu128 | timm: 1.0.11


## Config

`FEATURE_SOURCE` yang menentukan backbone beku: `'dinov2_large'` (default, direkomendasikan
duluan -- linear-probe DINOv2 secara umum unggul dibanding CLIP linear-probe untuk tugas
fine-grained di literatur) atau `'clip_vitb32'` (alternatif/pembanding, lebih ringan tapi
dilatih dengan objektif kontrastif teks-gambar yang punya alasan berbeda untuk cocok dengan
masalah kita -- lihat markdown di atas). Jalankan salah satu dulu; kalau waktu memungkinkan,
jalankan keduanya sebagai DUA anggota ensemble terpisah (`foundation_dinov2_large` dan
`foundation_clip_vitb32`) -- NB03 akan menimbang otomatis, bahkan bisa memberi bobot ~0 ke
salah satu kalau tidak membantu (guardrail PRD §10.2 yang sudah ada).


In [3]:
FEATURE_SOURCE = "dinov2_large"  # atau "clip_vitb32"

FEATURE_SPECS = {
    "dinov2_large": {
        "kind": "timm",
        "timm_tag": "vit_large_patch14_dinov2.lvd142m",
        "feat_dim": 1024 * 4,  # STRATEGI C: Multi-scale (4 layer terakhir digabung),
    },
    "clip_vitb32": {
        "kind": "clip",
        "hf_tag": "openai/clip-vit-base-patch32",
        "feat_dim": 512,
    },
}
SPEC = FEATURE_SPECS[FEATURE_SOURCE]
ARCH_NAME = f"foundation_{FEATURE_SOURCE}"

MASTER_DATASET_VERSION = 1  # harus sama dengan yang dipin di CONFIG 02a-02d (PRD §6.3)
# Notebook ini tidak berbagi kode dengan skeleton 02x (arsitektur trainingnya beda total: backbone
# beku + head kecil, bukan fine-tune penuh). String di bawah dipin SAMA PERSIS dengan SKELETON_VERSION
# di skeleton_body.txt murni sebagai stempel kompatibilitas "pakai fold_assignment.csv + master
# dataset versi yang sama" untuk gate integritas NB03 -- BUKAN klaim berbagi logika training.
SKELETON_VERSION_PIN = "skeleton-v1.3.0"

print(f"FEATURE_SOURCE={FEATURE_SOURCE} | ARCH_NAME={ARCH_NAME} | feat_dim={SPEC['feat_dim']}")


FEATURE_SOURCE=dinov2_large | ARCH_NAME=foundation_dinov2_large | feat_dim=1024


## Mount Drive & path (pola sama dengan `00_Diagnostik_Kemiripan_TrainTest.ipynb` / `02x`)


In [4]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Big Data Challenge - Satria Data 2026')
PROCESSED_ROOT = DRIVE_ROOT / 'Preprocessing_Output' / 'processed'
MANIFEST_DIR   = DRIVE_ROOT / 'Preprocessing_Output' / 'manifests'
OOF_ROOT       = DRIVE_ROOT / 'Preprocessing_Output' / 'oof'  # NB03 baca langsung dari sini, tidak perlu upload manual

# Folder sibling di dalam Training_Output yang sama dengan 4 arsitektur lain (EfficientNetV2-M,
# MaxViT, Swin Transformer V2, ConvNeXt V2) -- BUKAN root terpisah, supaya semua checkpoint model
# (termasuk anggota ke-5 ini) tetap terkumpul di satu tempat yang sama persis.
FOLDER_NAME_BY_SOURCE = {"dinov2_large": "DINOv2-Large", "clip_vitb32": "CLIP-ViT-B32"}
FOUNDATION_ROOT = DRIVE_ROOT / 'Training_Output' / FOLDER_NAME_BY_SOURCE[FEATURE_SOURCE]
FOUNDATION_ROOT.mkdir(parents=True, exist_ok=True)
OOF_ROOT.mkdir(parents=True, exist_ok=True)

FOLD_ASSIGNMENT_PATH = MANIFEST_DIR / 'fold_assignment.csv'
EVIDENCE_ROOT = DRIVE_ROOT / 'Preprocessing_Output' / 'diagnostik_hard_cases'
PLAN_CSV = EVIDENCE_ROOT / 'finetune_stage35_plan.csv'
RELABEL_CSV = EVIDENCE_ROOT / 'all_label_corrections.csv'
FEATURES_NPY = FOUNDATION_ROOT / 'train_features.npy'
FEATURE_IDS_NPY = FOUNDATION_ROOT / 'train_feature_ids.npy'

for p in [PROCESSED_ROOT, FOLD_ASSIGNMENT_PATH]:
    assert Path(p).exists(), f"path wajib tidak ditemukan: {p}"

fold_assignment = pd.read_csv(FOLD_ASSIGNMENT_PATH)
print(f"fold_assignment.csv: {len(fold_assignment)} baris, kolom {list(fold_assignment.columns)}")


Mounted at /content/drive
fold_assignment.csv: 26448 baris, kolom ['image_id', 'label', 'fold']


## Staging gambar ke disk lokal (hindari bottleneck Drive, pola sama dengan `02x`)

Ekstraksi fitur tetap satu pass penuh atas 26k gambar -- I/O per-file dari Drive yang ter-mount
jauh lebih lambat daripada dari disk lokal Colab, jadi gambar disalin sekali ke
`/content/local_staging_foundation/` dulu sebelum loop ekstraksi jalan (persis mekanisme
`parallel_copy` di `02x_Finetune_Stage35.ipynb`). Kalau cache fitur (`train_features.npy`) sudah
ada, staging ini otomatis dilewati -- tidak ada gunanya menyalin ulang gambar yang tidak akan dibaca.


In [5]:
LOCAL_ROOT = Path('/content/local_staging_foundation')
LOCAL_PROC = LOCAL_ROOT / 'processed'
LOCAL_PROC.mkdir(parents=True, exist_ok=True)


def parallel_copy(pairs, desc, n=64, timeout=60):
    def _one(s, d):
        d = Path(d)
        if d.exists() and d.stat().st_size > 0:
            return
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(s, d)
    errs = []
    with ThreadPoolExecutor(max_workers=n) as ex:
        futs = {ex.submit(_one, s, d): (s, d) for s, d in pairs}
        for f in tqdm(list(futs), total=len(futs), desc=desc):
            try:
                f.result(timeout=timeout)
            except Exception as e:
                errs.append(str(e))
    return errs


_features_cached = False
if FEATURES_NPY.exists() and FEATURE_IDS_NPY.exists():
    _saved_features = np.load(FEATURES_NPY, mmap_mode='r')
    if _saved_features.shape[1] == SPEC["feat_dim"]:
        _features_cached = len(np.load(FEATURE_IDS_NPY)) == len(fold_assignment)
    else:
        print(f"Dimensi fitur cache lama ({_saved_features.shape[1]}) usang. Menghapus cache lama...")
        FEATURES_NPY.unlink(missing_ok=True)
        FEATURE_IDS_NPY.unlink(missing_ok=True)

if _features_cached:
    print("Cache fitur sudah lengkap -- staging gambar lokal dilewati.")
else:
    pairs = [(PROCESSED_ROOT / f'{iid}.jpg', LOCAL_PROC / f'{iid}.jpg')
             for iid in fold_assignment['image_id']]
    errs = parallel_copy(pairs, 'salin processed ke lokal')
    if errs:
        print(f'PERINGATAN: {len(errs)} file gagal disalin')
    print(f'Staging selesai: {len(pairs)} gambar -> {LOCAL_PROC}')


def file_md5(path, chunk_size=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def hash_config(cfg: dict) -> str:
    return hashlib.sha256(json.dumps(cfg, sort_keys=True, default=str).encode()).hexdigest()[:16]


PROVENANCE = {
    "fold_assignment_md5": file_md5(FOLD_ASSIGNMENT_PATH),
    "master_dataset_version": MASTER_DATASET_VERSION,
    "skeleton_version": SKELETON_VERSION_PIN,
    "config_hash": hash_config({"ARCH_NAME": ARCH_NAME, "FEATURE_SOURCE": FEATURE_SOURCE, **SPEC}),
}
print("provenance:", PROVENANCE)


Cache fitur sudah lengkap -- staging gambar lokal dilewati.
provenance: {'fold_assignment_md5': '8489f6112e9e1960b912bb08cb16253b', 'master_dataset_version': 1, 'skeleton_version': 'skeleton-v1.3.0', 'config_hash': '2fe00aed81a855ad'}


## Bangun backbone beku + fungsi ekstraksi fitur

`torch.no_grad()` di seluruh forward pass -- tidak ada gradien yang pernah mengalir ke backbone.
`model.eval()` dipanggil sekali dan tidak pernah diubah balik ke `.train()` untuk backbone ini.


In [6]:
class RawImageIdDataset(Dataset):
    def __init__(self, image_ids, transform):
        self.image_ids = list(image_ids)
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = int(self.image_ids[idx])
        path = f"{LOCAL_PROC}/{image_id}.jpg"  # dibaca dari staging lokal, bukan Drive langsung (hindari bottleneck I/O)
        with Image.open(path) as im:
            tensor = self.transform(im.convert("RGB"))
        return tensor, image_id


if SPEC["kind"] == "timm":
    backbone = timm.create_model(SPEC["timm_tag"], pretrained=True, num_classes=0).to(DEVICE).eval()
    for p in backbone.parameters():
        p.requires_grad_(False)
    data_cfg = timm.data.resolve_data_config({}, model=backbone)
    extract_transform = timm.data.create_transform(**data_cfg, is_training=False)

    # STRATEGI C: Ambil output dari 4 block terakhir (multi-scale)
    intermediate_outputs = []
    def hook_fn(module, input, output):
        # output ViT blocks biasanya tuple atau tensor: (B, N, C). Kita ambil CLS token (indeks 0)
        cls_token = output[:, 0]
        intermediate_outputs.append(cls_token)

    n_blocks = len(backbone.blocks)
    for i in range(n_blocks - 4, n_blocks):
        backbone.blocks[i].register_forward_hook(hook_fn)

    @torch.no_grad()
    def extract_batch(images_tensor):
        intermediate_outputs.clear()
        _ = backbone(images_tensor.to(DEVICE))
        multi_scale_feat = torch.cat(intermediate_outputs, dim=1)
        return multi_scale_feat.cpu().numpy()

elif SPEC["kind"] == "clip":
    from transformers import CLIPModel, CLIPProcessor
    clip_model = CLIPModel.from_pretrained(SPEC["hf_tag"]).to(DEVICE).eval()
    clip_processor = CLIPProcessor.from_pretrained(SPEC["hf_tag"])
    for p in clip_model.parameters():
        p.requires_grad_(False)
    _clip_img_mean = clip_processor.image_processor.image_mean
    _clip_img_std = clip_processor.image_processor.image_std
    _clip_size = clip_processor.image_processor.crop_size["height"]

    import torchvision.transforms as T
    extract_transform = T.Compose([
        T.Resize(_clip_size, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(_clip_size),
        T.ToTensor(),
        T.Normalize(mean=_clip_img_mean, std=_clip_img_std),
    ])

    @torch.no_grad()
    def extract_batch(images_tensor):
        vision_out = clip_model.vision_model(pixel_values=images_tensor.to(DEVICE))
        feat = clip_model.visual_projection(vision_out.pooler_output)
        return F.normalize(feat, dim=1).cpu().numpy()

else:
    raise ValueError(f"FEATURE_SOURCE tidak dikenal: {FEATURE_SOURCE}")

print(f"Backbone {SPEC.get('timm_tag') or SPEC.get('hf_tag')} siap (frozen, eval mode).")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Backbone vit_large_patch14_dinov2.lvd142m siap (frozen, eval mode).


## Ekstraksi fitur train (resumable -- checkpoint tiap beberapa batch)

Ekstraksi 26k gambar makan waktu lama (single-GPU forward pass lewat backbone besar), dan sesi bisa
putus/ganti akun di tengah jalan. Supaya progres tidak hilang: setiap `SAVE_EVERY_N_BATCHES` batch,
fitur yang sudah diekstrak di-flush ke `train_features.npy`/`train_feature_ids.npy` di Drive. Saat
notebook dijalankan ulang (akun/sesi baru sekalipun, selama `FOUNDATION_ROOT` di Drive yang sama
ter-mount), sel ini otomatis membaca file cache tersebut, lanjut TEPAT dari gambar berikutnya yang
belum diekstrak -- bukan mulai dari nol. Urutan `all_ids` deterministik (dari `fold_assignment.csv`
yang dibekukan) jadi aman dipakai sebagai penanda progres.


In [7]:
SAVE_EVERY_N_BATCHES = 20  # ~20 batch @ batch_size 64 -> checkpoint tiap ~1280 gambar

all_ids = fold_assignment["image_id"].values
total_n = len(all_ids)

if FEATURES_NPY.exists() and FEATURE_IDS_NPY.exists():
    saved_features = np.load(FEATURES_NPY)
    saved_ids = np.load(FEATURE_IDS_NPY)
    if saved_features.shape[1] != SPEC["feat_dim"]:
        print(f"Dimensi fitur cache ({saved_features.shape[1]}) tidak cocok dengan SPEC ({SPEC['feat_dim']}). Cache dibatalkan!")
        saved_features = np.zeros((0, SPEC["feat_dim"]), dtype=np.float32)
        saved_ids = np.zeros((0,), dtype=np.int64)
else:
    saved_features = np.zeros((0, SPEC["feat_dim"]), dtype=np.float32)
    saved_ids = np.zeros((0,), dtype=np.int64)

n_done = len(saved_ids)

if n_done >= total_n:
    train_features, train_feature_ids = saved_features, saved_ids
    print(f"Cache fitur lengkap ditemukan: {train_features.shape} -- ekstraksi dilewati.")
else:
    assert np.array_equal(saved_ids, all_ids[:n_done]), (
        "Urutan image_id di fold_assignment.csv tidak cocok dengan cache parsial yang ada -- "
        "fold_assignment.csv berubah sejak checkpoint terakhir. Hapus train_features.npy/"
        "train_feature_ids.npy di FOUNDATION_ROOT dan mulai ulang dari nol."
    )
    remaining_ids = all_ids[n_done:]
    print(f"Melanjutkan ekstraksi dari checkpoint: {n_done}/{total_n} gambar sudah selesai sebelumnya.")

    ds = RawImageIdDataset(remaining_ids, extract_transform)
    # setelah staging ke disk lokal, bottleneck pindah dari I/O Drive ke CPU (decode+resize) --
    # pakai semua core yang tersedia (bukan angka tetap) supaya tidak under-utilize di runtime
    # dengan CPU lebih banyak (mis. Colab Pro/Pro+).
    n_workers = max(1, (os.cpu_count() or 2) - 1)
    loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=n_workers,
                         pin_memory=(DEVICE.type == "cuda"), persistent_workers=(n_workers > 0),
                         prefetch_factor=4 if n_workers > 0 else None)
    print(f"DataLoader ekstraksi memakai {n_workers} worker (os.cpu_count()={os.cpu_count()})")

    new_features, new_ids = [], []

    def flush_checkpoint():
        feats = np.concatenate([saved_features] + new_features, axis=0) if new_features else saved_features
        ids = np.concatenate([saved_ids] + new_ids, axis=0) if new_ids else saved_ids
        np.save(FEATURES_NPY, feats)
        np.save(FEATURE_IDS_NPY, ids)
        return feats, ids

    for batch_i, (images, ids) in enumerate(tqdm(loader, desc=f"Ekstraksi fitur {FEATURE_SOURCE}")):
        feats = extract_batch(images)
        new_features.append(feats)
        new_ids.append(ids.numpy())
        if (batch_i + 1) % SAVE_EVERY_N_BATCHES == 0:
            flush_checkpoint()
            print(f"  checkpoint: {n_done + sum(len(x) for x in new_ids)}/{total_n} gambar tersimpan")

    train_features, train_feature_ids = flush_checkpoint()
    print(f"Ekstraksi selesai: {train_features.shape} disimpan ke {FEATURES_NPY}")

feature_lookup = dict(zip(train_feature_ids.tolist(), range(len(train_feature_ids))))
fold_lookup = dict(zip(fold_assignment["image_id"], fold_assignment["fold"]))

# Semuanya menggunakan label_lookup_train (yang sudah dikoreksi) karena kita
# sudah resmi bergeser ke Finetune Stage 3.5 untuk semua arsitektur.
label_lookup_train = dict(zip(fold_assignment["image_id"], fold_assignment["label"]))

if RELABEL_CSV.exists():
    relabel_df = pd.read_csv(RELABEL_CSV)
    relabel_map = dict(zip(relabel_df["image_id"], relabel_df["expected_label"]))
    n_corrected = sum(1 for iid in label_lookup_train if iid in relabel_map)
    label_lookup_train.update({iid: relabel_map[iid] for iid in label_lookup_train if iid in relabel_map})
    print(f"Label correction diterapkan: {n_corrected} gambar (true_label OOF juga ikut terkoreksi)")
else:
    print("all_label_corrections.csv tidak ditemukan -- training pakai label mentah juga")

if PLAN_CSV.exists():
    plan_df = pd.read_csv(PLAN_CSV)
    oversample_map = dict(zip(plan_df["image_id"], plan_df["oversample_weight"]))
    print(f"Oversample plan dimuat: {len(oversample_map)} gambar (bobot loss per-sample, bukan resampling)")
else:
    oversample_map = {}
    print("finetune_stage35_plan.csv tidak ditemukan -- tidak ada oversampling diterapkan")


Cache fitur lengkap ditemukan: (26448, 1024) -- ekstraksi dilewati.
Label correction diterapkan untuk training: 20 gambar (true_label OOF tetap mentah)
Oversample plan dimuat: 269 gambar (bobot loss per-sample, bukan resampling)


## Head kecil (MLP 2 layer) di atas fitur beku

Ini satu-satunya bagian yang benar-benar "dilatih" per fold. Input sudah berupa vektor tetap
(bukan gambar) jadi tidak ada I/O gambar maupun forward-pass backbone di loop ini sama sekali --
karena itu satu fold selesai dalam hitungan detik-menit walau di CPU sekalipun.


In [8]:
class FeatureHead(nn.Module):
    def __init__(self, in_dim, hidden=512, num_classes=3, p_drop=0.2):
        super().__init__()
        # STRATEGI A: Deeper Head (3-layer) + LayerNorm
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden, hidden // 2),
            nn.LayerNorm(hidden // 2),
            nn.GELU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden // 2, num_classes),
        )

    def forward(self, x):
        return self.net(x)


def make_fold_tensors(fold_k, label_map):
    train_ids = fold_assignment.loc[fold_assignment["fold"] != fold_k, "image_id"].values
    val_ids = fold_assignment.loc[fold_assignment["fold"] == fold_k, "image_id"].values

    def _gather(ids):
        idx = [feature_lookup[i] for i in ids]
        X = torch.from_numpy(train_features[idx]).float()
        y = torch.tensor([label_map[i] for i in ids], dtype=torch.long)
        return X, y, ids

    return _gather(train_ids), _gather(val_ids)


def build_sample_weights(ids, default_w=1.0):
    return torch.tensor([oversample_map.get(int(i), default_w) for i in ids], dtype=torch.float32)


@torch.no_grad()
def eval_head(head, X, y):
    head.eval()
    logits = head(X.to(DEVICE))
    probs = F.softmax(logits, dim=1).cpu().numpy()
    preds = probs.argmax(1)
    macro_f1 = f1_score(y.numpy(), preds, average="macro")
    return macro_f1, probs


HEAD_EPOCHS = 300
HEAD_LR = 1e-3
HEAD_HIDDEN = 512
HEAD_DROPOUT = 0.2
PATIENCE = 30


## Loop 5 fold -- latih head, evaluasi, ekspor OOF sesuai skema kanonis PRD §5.3


In [9]:
results_per_fold = []

for fold_k in range(N_FOLDS):
    oof_path = OOF_ROOT / f"oof_{ARCH_NAME}_fold{fold_k}.csv"
    if oof_path.exists():
        print(f"[fold {fold_k}] OOF sudah ada -- dilewati")
        continue

    (X_train, y_train, train_ids), (X_val, y_val, val_ids) = make_fold_tensors(fold_k, label_lookup_train)
    X_train, y_train = X_train.to(DEVICE), y_train.to(DEVICE)
    train_weights = build_sample_weights(train_ids).to(DEVICE)
    n_weighted_in_train = int((train_weights != 1.0).sum())
    print(f"Gambar dari plan (oversample) di train set: {n_weighted_in_train}")

    torch.manual_seed(SEED + fold_k)
    head = FeatureHead(SPEC["feat_dim"], HEAD_HIDDEN, NUM_CLASSES, HEAD_DROPOUT).to(DEVICE)
    optimizer = torch.optim.AdamW(head.parameters(), lr=HEAD_LR, weight_decay=1e-2)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1, reduction="none")

    best_f1 = -1.0
    best_state = None
    epochs_since_improve = 0

    for epoch in range(1, HEAD_EPOCHS + 1):
        head.train()
        optimizer.zero_grad(set_to_none=True)
        per_sample_loss = criterion(head(X_train), y_train)
        loss = (per_sample_loss * train_weights).sum() / train_weights.sum()
        loss.backward()
        optimizer.step()

        val_f1, _ = eval_head(head, X_val, y_val)
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.clone() for k, v in head.state_dict().items()}
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1
            if epochs_since_improve >= PATIENCE:
                break

    head.load_state_dict(best_state)
    final_f1, final_probs = eval_head(head, X_val, y_val)
    print(f"[fold {fold_k}] epoch berhenti={epoch} | val macro-F1 terbaik={final_f1:.5f}")

    oof_rows = []
    for i, image_id in enumerate(val_ids):
        p = final_probs[i]
        oof_rows.append({
            "image_id": int(image_id), "prob_0": float(p[0]), "prob_1": float(p[1]), "prob_2": float(p[2]),
            "true_label": int(label_lookup_raw[image_id]), "fold": fold_k,
        })
    oof_df = pd.DataFrame(oof_rows).sort_values("image_id").reset_index(drop=True)
    oof_df.to_csv(oof_path, index=False)

    sidecar = {**PROVENANCE, "arch": ARCH_NAME, "fold": fold_k, "val_macro_f1": float(final_f1),
               "head_hidden": HEAD_HIDDEN, "head_dropout": HEAD_DROPOUT}
    with open(OOF_ROOT / f"oof_{ARCH_NAME}_fold{fold_k}.json", "w") as f:
        json.dump(sidecar, f, indent=2)

    fold_ckpt_dir = FOUNDATION_ROOT / f"fold{fold_k}"
    fold_ckpt_dir.mkdir(parents=True, exist_ok=True)
    torch.save({
        "head_state": best_state, "feat_dim": SPEC["feat_dim"], "hidden": HEAD_HIDDEN,
        "feature_source": FEATURE_SOURCE, "val_macro_f1": float(final_f1), **PROVENANCE,
    }, fold_ckpt_dir / "head.ckpt")

    results_per_fold.append(final_f1)
    print(f"menulis {oof_path}")

if results_per_fold:
    print(f"\n{ARCH_NAME}: {len(results_per_fold)} fold baru selesai, rata-rata val macro-F1={np.mean(results_per_fold):.5f}")
else:
    print(f"\n{ARCH_NAME}: semua 5 fold OOF sudah ada sebelumnya, tidak ada yang dilatih ulang.")


Gambar dari plan (oversample) di train set: 213
[fold 0] epoch berhenti=144 | val macro-F1 terbaik=0.98003
menulis /content/drive/MyDrive/Big Data Challenge - Satria Data 2026/Preprocessing_Output/oof/oof_foundation_dinov2_large_fold0.csv
Gambar dari plan (oversample) di train set: 220
[fold 1] epoch berhenti=146 | val macro-F1 terbaik=0.98159
menulis /content/drive/MyDrive/Big Data Challenge - Satria Data 2026/Preprocessing_Output/oof/oof_foundation_dinov2_large_fold1.csv
Gambar dari plan (oversample) di train set: 216
[fold 2] epoch berhenti=139 | val macro-F1 terbaik=0.97777
menulis /content/drive/MyDrive/Big Data Challenge - Satria Data 2026/Preprocessing_Output/oof/oof_foundation_dinov2_large_fold2.csv
Gambar dari plan (oversample) di train set: 218
[fold 3] epoch berhenti=207 | val macro-F1 terbaik=0.98058
menulis /content/drive/MyDrive/Big Data Challenge - Satria Data 2026/Preprocessing_Output/oof/oof_foundation_dinov2_large_fold3.csv
Gambar dari plan (oversample) di train set: 

## Langkah lanjutan (belum dilakukan di sini, catatan untuk sesi berikutnya)

1. Cek `oof_report.md`/bobot Nelder-Mead di NB03 setelah menambahkan `"{ARCH_NAME}"` ke daftar
   `ARCHS` -- kalau bobotnya mendekati nol, itu sinyal sah bahwa anggota ini tidak menambah nilai
   untuk data ini (guardrail PRD §10.2 yang sudah ada, bukan kegagalan proses).
2. Kalau bobotnya positif dan cukup besar: perlu pembaruan NB04 (satu-satunya notebook yang boleh
   membuka `test/`) untuk (a) ekstraksi fitur test dengan backbone beku yang sama, (b) memuat
   `head.ckpt` per fold dari `Training_Output_Foundation/{ARCH_NAME}/foldK/`, (c) rata-rata
   probabilitas 5 fold seperti anggota lain -- ini BELUM diimplementasikan di notebook manapun,
   sengaja ditunda supaya PRD §11 (test hanya disentuh NB04) tidak pernah dilanggar di sini.
3. Opsional: ulangi notebook ini dengan `FEATURE_SOURCE = "clip_vitb32"` sebagai anggota keenam
   yang terpisah (`foundation_clip_vitb32`), lalu biarkan NB03 menimbang keduanya sekaligus.
